<a href="https://colab.research.google.com/github/rounak393/clab/blob/main/clab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.8 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader

import medmnist
from medmnist import INFO

In [3]:
data_flag = 'pathmnist'
download = True

info = INFO[data_flag]
DataClass = getattr(medmnist, info['python_class'])

train_dataset = DataClass(split='train', download=download)
test_dataset = DataClass(split='test', download=download)

100%|██████████| 206M/206M [00:11<00:00, 18.0MB/s]


In [4]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
])

train_dataset.transform = transform
test_dataset.transform = transform

In [5]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [6]:
class FusionModel(nn.Module):

    def __init__(self, num_classes=9):
        super().__init__()

        self.mobilenet = models.mobilenet_v3_small(weights="DEFAULT")
        self.mobilenet.classifier = nn.Identity()

        self.efficientnet = models.efficientnet_b0(weights="DEFAULT")
        self.efficientnet.classifier = nn.Identity()

        self.fc = nn.Sequential(
            nn.Linear(576 + 1280, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        f1 = self.mobilenet(x)
        f2 = self.efficientnet(x)

        fused = torch.cat((f1, f2), dim=1)

        out = self.fc(fused)

        return out

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FusionModel().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 104MB/s]


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 161MB/s]


In [8]:
for param in model.mobilenet.parameters():
    param.requires_grad = False

for param in model.efficientnet.parameters():
    param.requires_grad = False

In [9]:
epochs = 5

for epoch in range(epochs):

    model.train()
    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.squeeze().to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {running_loss/len(train_loader)}")

Epoch 1 Loss: 0.5609365623939435
Epoch 2 Loss: 0.36207600419264613
Epoch 3 Loss: 0.32611226496454154
Epoch 4 Loss: 0.30448735516462755
Epoch 5 Loss: 0.2868779731997803


In [10]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.squeeze().to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", 100 * correct / total)

Test Accuracy: 88.32869080779945
